# Setup Environment

In [ ]:
%load_ext autoreload
%autoreload 1

%aimport highlighting_precision
%aimport gradcam_APs
%aimport gradcam
%aimport vanilla_vgg
%aimport dataHelper
%aimport external

import warnings
from torch.serialization import SourceChangeWarning
warnings.filterwarnings("ignore", category=SourceChangeWarning)

%matplotlib inline
!nvidia-smi

In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ext = external.ExternalValidationContext(
    environment = 'icad',
    external_validation_path=Path('/redacted/path/to/ext_val'),
    data_path=Path('/redacted/path/to/data') / 'icad-test',
    models=['iaia-bl', 'protopnet',  'vgg16'],
    
    #NOTE: This margin_name_labels are unique to ICAD
    margin_name_labels = [
        (0, 'circ_prob+indst_prob', 'not_spiculated'),
        (1, 'spic_prob', 'spiculated')
    ],
    malignancy_name_labels = [
        (0, 'mal_neg_prob', 'not_malignant'),
        (1, 'mal_pos_prob', 'malignant')
    ],
    subgroupings  = [
        ('all', 'all'), 
        ('manufacturer', 'ge'), ('manufacturer', 'hologic'), ('manufacturer', 'siemens'),
        ('age_3_bucket', '<50'), ('age_3_bucket','50-69'), ('age_3_bucket', '>70'),
        ('age_2_bucket', '<=57'), ('age_2_bucket','>57')
        #No Race on ICAD    
    ],
    classmap = {
        'Circumscribed': 0,
        'Indistinct': 0,
        'Spiculated': 1,
        
        'not_spiculated': 0,
        'spiculated': 1
    }
)

ext

# ICAD Validation

## Activation Precision

### GradCAM for VGG16

In [ ]:
gradcam_aps = ext.run_gradcams(num_bootstraps=3)
gradcam_aps.pipe(display)
gradcam_aps.groupby(['method']).mean().pipe(display)

#### ProtoPNets

In [ ]:
from highlighting_precision import hp
import numpy as np
import torch
import scipy.stats as st

def icad_classwise_activation_precision_for_protopnets(num_bootstraps=3):
    # World be better if the existing activation precision method was generalized to handle this case, but for now, this works.
    np.random.seed(0)
    torch.random.manual_seed(0)
    ap_ci_df = pd.DataFrame()

    for model_name, m in [('iaia-bl', ext.iaiabl_path), ('protopnet', ext.protopnet_path)]:
        print(model_name, "-------------")

        res_list_protop = hp(ext.data_path, 
                m,
                per_proto=True,
                num_bootstraps=num_bootstraps,
                classmap=ext.classmap,
                verbose=False)
        
        # Average across prototypes within a class
        res_list_protop_np = np.array(res_list_protop)

        other_indices = res_list_protop_np[:, :, 0] <= 9
        spic_indices = res_list_protop_np[:, :, 0] > 9
        other_aps = np.mean(res_list_protop_np[other_indices, 1].reshape(-1, 5), axis=0)
        spic_aps = np.mean(res_list_protop_np[spic_indices, 1].reshape(-1, 5), axis=0)
        
        # create 95% confidence interval
        for ap_list, margin in [(other_aps, 'not_spiculated'), (spic_aps, 'spiculated')]:

            mean_ap = np.mean(ap_list)
            ci = st.t.interval(alpha=0.95, df=len(ap_list)-1,
                loc=mean_ap,
                scale=st.sem(ap_list))

            ap_ci_df = pd.concat([ap_ci_df, pd.DataFrame({
                'ap': [mean_ap],
                'ci_025': [ci[0]],
                'ci_975': [ci[1]],
                'model': [model_name],
                'margin': [margin],
                'num_bootstraps': [num_bootstraps]
            })])

    return ap_ci_df

In [ ]:
protopnets_aps = icad_classwise_activation_precision_for_protopnets(num_bootstraps=3)
protopnets_aps.pipe(display)
protopnets_aps.groupby(['model']).mean().pipe(display)

# Model Predictions
## Margin Predictions

In [ ]:
margin_predictions = ext.margin_predictions_for_models(image_has_rois=True)
margin_predictions.to_csv(ext.external_validation_path/f'{ext.environment}_margin_predictions.csv', index=False)
margin_predictions.groupby('model_name').count()

In [ ]:
__prob_sums = margin_predictions[['circ_prob', 'indst_prob', 'spic_prob']].sum(axis=1).round(2).value_counts()
assert len(__prob_sums) == 1 and 1.0 in __prob_sums
__prob_sums

## Analyze Results

In [ ]:
predictions = pd.read_csv(ext.external_validation_path/f'{ext.environment}_margin_predictions.csv')
predictions.head(3).pipe(display)

In [ ]:
ext.plot_margin_probs(predictions)

## Malignancy Prediction
### Create a Dataframes with Malignancy Labels

The original dataset had some diagnostic mammos in it, but from here on it's all screening.

In [ ]:
test_data_path = Path('/redacted/path/to/test/data')
clean_df = pd.read_csv(test_data_path / 'clean_df.csv')
clean_df.head(3)

In [ ]:
# Standardize labeling scheme
predictions['mass_uid'] = predictions['patient_id'].str.replace('_', '.')
predictions['patient_mass_id'] = predictions['patient_id']
predictions_with_labels = predictions.drop('patient_id', axis=1).merge(clean_df, on='mass_uid')

pred_cols = ['circ_logit',  'indst_logit',  'spic_logit', 'circ_prob', 'indst_prob', 'spic_prob', 'context_size', 'image_id', 'model_name', 'mal_neg_prob',
       'mal_pos_prob'] + ['mass_uid', 'margin_label']
predictions_with_labels['histopath'] = predictions_with_labels['mass_histo']

predictions_with_labels['margin_label'] =predictions_with_labels['mass_margin'].apply(lambda x:  0 if x != 'spiculated' else 1)
pred_cols.remove('image_id')

print(predictions.shape)
predictions_with_labels.head(3)[pred_cols]

### Run Malignancy Prediction

In [ ]:
## predictions_wmal Malignancy Predictions
predictions_wmal = ext.malignancy_for_models(predictions_with_labels)

predictions_wmal.groupby('model_name').count()[['mal_pos_prob', 'mal_neg_prob', 'malignancy_pred']]

In [ ]:
__prob_sums = predictions_wmal[['mal_pos_prob', 'mal_neg_prob']].sum(axis=1).round(2).value_counts()
assert len(__prob_sums) == 1 and 1.0 in __prob_sums
__prob_sums

In [ ]:
for model in ext.models:
    predictions_wmal[predictions_wmal['model_name'] == model]['mal_pos_prob'].hist(alpha=0.5)

# Analysis

## Merge in Patient Age

In [ ]:
age_df = pd.DataFrame()

for csv in (ext.external_validation_path / 'age').iterdir():
    print('reading', csv)
    this_df = pd.read_csv(csv, sep=' ', header=None, names=['patient_id', 'age'])
    age_df = pd.concat([age_df, this_df])

age_df.head().pipe(display)
age_df['patient_id'].nunique()

In [ ]:
predictions_wmal_labels_age = predictions_wmal.merge(age_df, how='left', on='patient_id')
predictions_wmal_labels_age.shape, predictions_wmal_labels_age.shape

In [ ]:
predictions_wmal_labels_age['age'].hist(bins=[0, 50, 70, 100])

### Final Demographics

In [ ]:
predictions_wmal_labels_age['image_id'] = predictions_wmal_labels_age['dcm_id']

In [ ]:
predictions_wmal_labels_age['age_3_bucket'] = predictions_wmal_labels_age['age'].apply(external.age_3_bucket)
predictions_wmal_labels_age['age_2_bucket'] = predictions_wmal_labels_age['age'].apply(external.age_2_bucket)

In [ ]:
summary = ext.patient_summary_statistics(predictions_wmal_labels_age)[['subgroup', 'value', 'count']]
summary.to_csv(f'{ext.environment}_summary.csv')
summary

# Export Key Results

In [ ]:
export_df = predictions_wmal_labels_age.copy()

## Export ROC

In [ ]:
ext.margin_name_labels[1:]

In [ ]:
margin_roc_df, margin_ci_df = ext.roc_dfs(export_df, 'margin', ext.margin_name_labels[1:], 'margin_label')
margin_roc_df.head(10).pipe(display)
margin_ci_df.pipe(display)

In [ ]:
margin_roc_df.to_csv(ext.external_validation_path / f'{ext.environment}_margin_roc.csv', index=False)
margin_roc_df.groupby(['model', 'margin']).count().pipe(display)

margin_ci_df.to_csv(ext.external_validation_path / f'{ext.environment}_margin_ci.csv', index=False)
margin_ci_df[margin_ci_df['subgrouping'] == 'all'].pivot(index='margin', columns='model', values=['auc', 'ci_025', 'ci_975']).swaplevel(0,1,axis=1)[ext.models].round(2)

# Note the warnings come from the fact some subgroups only have one entry in the class. Obviously, that is not a valid test, but it's included for completeness.

In [ ]:
# ICAD Specific Margin Plot
plt.figure()

# Plot ROC curve for each class
for model in ext.models:
    class_data = margin_roc_df[(margin_roc_df['model'] == model) & (margin_roc_df['margin'] == 'spiculated')].sort_values('threshold')

    row = margin_ci_df[(margin_ci_df['subgrouping'] == 'all') & (margin_ci_df['model'] == model) & (margin_ci_df['margin'] == 'spiculated')].round(2)

    auc_label = f'AUC: {row["auc"].item()} ({row["ci_025"].item()}, {row["ci_975"].item()})'

    plt.plot(class_data['fpr'], class_data['tpr'], label=f'Model {model}\n{auc_label}')

plt.plot([0, 1], [0, 1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Spiculated Margins by Model')
plt.legend(loc='best')

# Show the plot
plt.show()

## Activation Precision

In [ ]:
EXPORT_BOOTSTRAPS=100

In [ ]:
prototype_ap_df = icad_classwise_activation_precision_for_protopnets(num_bootstraps=EXPORT_BOOTSTRAPS)
prototype_ap_df

In [ ]:
gradcam_ap_df = ext.run_gradcams(num_bootstraps=EXPORT_BOOTSTRAPS)
gradcam_ap_df

In [ ]:
ap_ci_df = pd.concat([prototype_ap_df, gradcam_ap_df], sort=True)
ap_ci_df.to_csv(ext.external_validation_path / f'{ext.environment}_ap_ci.csv', index=False)
ap_ci_df.fillna('').groupby(['model', 'method']).mean().pipe(display)
ap_ci_df.pipe(display)

### Malignancy Results

In [ ]:
export_df[['mal_pos_prob', 'malignancy_label']].to_csv(ext.external_validation_path/f'{ext.environment}_mal_dist.csv', index=False)

In [ ]:
mal_roc_df, mal_ci_df = ext.roc_dfs(export_df, 'malignancy', ext.malignancy_name_labels, 'malignancy_label')
mal_roc_df.pipe(display)
mal_ci_df.pipe(display)

In [ ]:
mal_roc_df.to_csv(ext.external_validation_path / f'{ext.environment}_mal_roc.csv', index=False)
mal_roc_df.groupby(['model', 'malignancy']).count().pipe(display)

mal_ci_df.to_csv(ext.external_validation_path / f'{ext.environment}_mal_ci.csv', index=False)
mal_ci_df[mal_ci_df['subgrouping'] == 'all']\
    .pivot(index='malignancy', columns='model', values=['auc', 'ci_025', 'ci_975'])\
    .swaplevel(0,1,axis=1)[ext.models].round(2)

# Note the warnings come from the fact some subgroups only have one entry in the class. Obviously, that is not a valid test, but it's included for completeness.

In [ ]:
plt.figure()

# Plot ROC curve for each class
for model in ext.models:
    class_data = mal_roc_df[(mal_roc_df['model'] == model) & (mal_roc_df['malignancy'] == 'malignant')].sort_values('threshold')

    row = mal_ci_df[(mal_ci_df['subgrouping'] == 'all') & (mal_ci_df['model'] == model) & (mal_ci_df['malignancy'] == 'malignant')].round(2)

    auc_label = f'AUC: {row["auc"].item()} ({row["ci_025"].item()}, {row["ci_975"].item()})'

    plt.plot(class_data['fpr'], class_data['tpr'], label=f'Model {model}\n{auc_label}')

plt.plot([0, 1], [0, 1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Malignancy')
plt.legend(loc='best')

# Show the plot
plt.show()